# 0.1 Import Libraries

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 0.2 Load Project Modules

In [ ]:
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from config.settings import PROCESSED_DIR, OUTPUT_DIR
from src.extrapolation import naive_extrapolation, stratified_extrapolation, contribution_weighted_extrapolation, category_level_extrapolation, compare_market_estimates
from src.evaluation import method_comparison_table, weekly_error_summary, category_level_error_summary

# 1.1 Load Universe Sales

In [ ]:
universe = pd.read_csv(OUTPUT_DIR / "active_store_universe.csv")
weekly_sales = pd.read_csv(PROCESSED_DIR / "weekly_store_sales.csv", parse_dates=["week"])
weekly_category_sales = pd.read_csv(PROCESSED_DIR / "weekly_store_category_sales.csv", parse_dates=["week"])
weekly_sales.head()

# 1.2 Load Sample Panels

In [ ]:
panel_store_list = pd.read_csv(OUTPUT_DIR / "sample_panel_store_list.csv")
panel = panel_store_list[panel_store_list["panel_name"] == "optimized_panel"].copy()
panel.head()

# 2.1 Naive Extrapolation

In [ ]:
naive_estimates = naive_extrapolation(weekly_sales, universe, panel)
naive_estimates.head()

# 2.2 Stratified Extrapolation

In [ ]:
stratified_estimates = stratified_extrapolation(weekly_sales, universe, panel)
stratified_estimates.head()

# 2.3 Contribution-Weighted Extrapolation

In [ ]:
weighted_estimates = contribution_weighted_extrapolation(weekly_sales, universe, panel)
weighted_estimates.head()

# 2.4 Category-Level Extrapolation

In [ ]:
category_estimates = category_level_extrapolation(weekly_category_sales, universe, panel)
category_estimates.head()

# 3.1 Evaluate Total Market Estimates

In [ ]:
weekly_estimates = pd.concat([naive_estimates, stratified_estimates, weighted_estimates], ignore_index=True)
weekly_estimates = compare_market_estimates(weekly_estimates)
method_comparison = method_comparison_table(weekly_estimates)
method_comparison

# 3.2 Evaluate Weekly Estimates

In [ ]:
weekly_error = weekly_error_summary(weekly_estimates)
weekly_error.head()

# 3.3 Evaluate Category-Level Estimates

In [ ]:
category_error = category_level_error_summary(category_estimates)
category_error.head()

# 3.4 Compare Bias by Method

In [ ]:
method_comparison[["method", "bias_pct", "wape", "mape"]]

# 4.1 Save Extrapolation Outputs

In [ ]:
method_comparison.to_csv(OUTPUT_DIR / "extrapolation_method_comparison.csv", index=False)
weekly_error.to_csv(OUTPUT_DIR / "weekly_extrapolation_error.csv", index=False)
category_error.to_csv(OUTPUT_DIR / "category_extrapolation_error.csv", index=False)
method_comparison